In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries imported successfully")

## 1. Generate Synthetic Training Data

Since we don't have historical data yet, we'll generate synthetic data based on domain knowledge.

In [ ]:
# Generate synthetic data
np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    'risk_score': np.random.uniform(0, 1, n_samples),
    'season': np.random.choice(['monsoon', 'summer', 'winter'], n_samples),
    'historical_claims': np.random.poisson(3, n_samples),
    'working_days_per_week': np.random.choice([5, 6, 7], n_samples),
    'coverage_amount': np.random.choice([600, 800, 1000], n_samples),
    'user_tenure_months': np.random.randint(0, 24, n_samples),
    'previous_fraud_score': np.random.uniform(0, 0.5, n_samples)
})

# Generate target premium (rule-based)
def calculate_premium(row):
    base = 40
    premium = base
    
    # Risk adjustment
    premium += row['risk_score'] * 15
    
    # Seasonal adjustment
    if row['season'] == 'monsoon':
        premium += 5
    elif row['season'] == 'summer':
        premium -= 2
    
    # Historical claims
    if row['historical_claims'] > 5:
        premium += 3
    
    # Tenure discount
    if row['user_tenure_months'] >= 6:
        premium -= 3
    elif row['user_tenure_months'] >= 3:
        premium -= 2
    
    # Fraud penalty
    premium += row['previous_fraud_score'] * 10
    
    # Add noise
    premium += np.random.normal(0, 2)
    
    # Bounds
    return np.clip(premium, 35, 80)

data['weekly_premium'] = data.apply(calculate_premium, axis=1)

print(f"Generated {len(data)} samples")
print(data.head())

## 2. Feature Engineering

In [ ]:
# One-hot encode categorical features
data_encoded = pd.get_dummies(data, columns=['season'], drop_first=False)

# Features and target
feature_columns = [col for col in data_encoded.columns if col != 'weekly_premium']
X = data_encoded[feature_columns]
y = data_encoded['weekly_premium']

print(f"Features: {feature_columns}")
print(f"Target: weekly_premium")
print(f"Shape: {X.shape}")

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 4. Train XGBoost Model

In [ ]:
# Train XGBoost regressor
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

print("✅ Model trained successfully")

## 5. Model Evaluation

In [ ]:
# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Metrics
print("Training Metrics:")
print(f"MAE: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.2f}")
print(f"R²: {r2_score(y_train, y_pred_train):.3f}")

print("\nTest Metrics:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"R²: {r2_score(y_test, y_pred_test):.3f}")

## 6. Feature Importance

In [ ]:
# Plot feature importance
importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='importance', y='feature')
plt.title('Feature Importance for Premium Pricing')
plt.tight_layout()
plt.show()

print(importance)

## 7. Save Model

In [ ]:
# Save model
model_path = '../backend/app/ml_models/pricing_model.pkl'
joblib.dump(model, model_path)

# Save feature names
feature_path = '../backend/app/ml_models/pricing_features.pkl'
joblib.dump(feature_columns, feature_path)

print(f"✅ Model saved to {model_path}")
print(f"✅ Features saved to {feature_path}")

## 8. Test Prediction

In [ ]:
# Test single prediction
test_input = pd.DataFrame([{
    'risk_score': 0.6,
    'historical_claims': 2,
    'working_days_per_week': 6,
    'coverage_amount': 800,
    'user_tenure_months': 6,
    'previous_fraud_score': 0.1,
    'season_monsoon': 1,
    'season_summer': 0,
    'season_winter': 0
}])

prediction = model.predict(test_input)[0]
print(f"\nPredicted weekly premium: ₹{prediction:.2f}")